# Agent的高级用法4：流式输出及模式

## 1、values输出模式

当 stream_mode 设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都要获取完整状态、状态持久化场景。


In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}

@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }

# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

for chunk in customer_service_agent.stream(
    {
        "messages": [
            {"role":"user", "content":"查询客户id为cust1234的完整信息，历史订单和可用优惠"}
        ]
    },
    stream_mode="values"
):
    rprint(chunk)
    print("-" * 50)

{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整信息，历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='178fae2a-de71-4441-a8e7-b4d5d1fe746c'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整信息，历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='178fae2a-de71-4441-a8e7-b4d5d1fe746c'
        ),
        AIMessage(
            content='我来帮您查询客户cust1234的完整信息，包括基本信息、历史订单和当前可用优惠。\n\n',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 104,
                    'prompt_tokens': 485,
                    'total_tokens': 589,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 485}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-fd72519c-9fa6-9074-a132-17524273d634',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6c0f-aaa3-7201-a229-aae59b64255a-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_d1b154e886434c288a6c5512',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_5bcc4c34a2fb40e3b0ab1889',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_85a607fac99d44e88650f012',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 485,
                'output_tokens': 104,
                'total_tokens': 589,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整信息，历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='178fae2a-de71-4441-a8e7-b4d5d1fe746c'
        ),
        AIMessage(
            content='我来帮您查询客户cust1234的完整信息，包括基本信息、历史订单和当前可用优惠。\n\n',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 104,
                    'prompt_tokens': 485,
                    'total_tokens': 589,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 485}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-fd72519c-9fa6-9074-a132-17524273d634',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6c0f-aaa3-7201-a229-aae59b64255a-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_d1b154e886434c288a6c5512',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_5bcc4c34a2fb40e3b0ab1889',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_85a607fac99d44e88650f012',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 485,
                'output_tokens': 104,
                'total_tokens': 589,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='0ecce049-c726-4bad-be65-fea49d9669e4',
            tool_call_id='call_d1b154e886434c288a6c5512'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='e7acc8fe-f53d-4772-8006-31c8decb28aa',
            tool_call_id='call_5bcc4c34a2fb40e3b0ab1889'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='a553c11a-5d6a-431e-a3ec-26acf2ef99ea',
            tool_call_id='call_85a607fac99d44e88650f012'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整信息，历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='178fae2a-de71-4441-a8e7-b4d5d1fe746c'
        ),
        AIMessage(
            content='我来帮您查询客户cust1234的完整信息，包括基本信息、历史订单和当前可用优惠。\n\n',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 104,
                    'prompt_tokens': 485,
                    'total_tokens': 589,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 485}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-fd72519c-9fa6-9074-a132-17524273d634',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6c0f-aaa3-7201-a229-aae59b64255a-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_d1b154e886434c288a6c5512',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_5bcc4c34a2fb40e3b0ab1889',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_85a607fac99d44e88650f012',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 485,
                'output_tokens': 104,
                'total_tokens': 589,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='0ecce049-c726-4bad-be65-fea49d9669e4',
            tool_call_id='call_d1b154e886434c288a6c5512'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='e7acc8fe-f53d-4772-8006-31c8decb28aa',
            tool_call_id='call_5bcc4c34a2fb40e3b0ab1889'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='a553c11a-5d6a-431e-a3ec-26acf2ef99ea',
            tool_call_id='call_85a607fac99d44e88650f012'
        ),
        AIMessage(
            content='根据查询结果，客户cust1234的完整信息如下：\n\n## 客户基本信息\n- **姓名**：张三\n- 
**会员等级**：VIP\n- **加入日期**：2023年1月15日\n\n## 订单历史\n- **总订单数**：15单\n- 
**总消费金额**：¥25,800.00\n\n## 当前可用优惠\n- **优惠活动**：\n  - 老用户优惠\n  - 会员专属折扣\n- 
**有效期**：至2027年1月31日\n\n该客户是一位VIP会员，自2023年初加入以来已有15笔订单，累计消费超过2.5万元，目前可以享
受老用户优惠和会员专属折扣两项促销活动。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 170,
                    'prompt_tokens': 689,
                    'total_tokens': 859,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 689}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-1d1f19c1-3d6c-99cf-9d64-6b15b780aa2a',
                'finish_reason': 'stop',
                'logprobs': None
            },

--------------------------------------------------


## 2、updates输出格式

这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。


In [3]:
for chunk in customer_service_agent.stream(
    {
        "messages": [
            {"role":"user", "content":"查询客户id为cust1234的完整信息，历史订单和可用优惠"}
        ]
    },
    stream_mode="updates"
):
    rprint(chunk)
    print("-" * 50)

{
    'model': {
        'messages': [
            AIMessage(
                content='我将为您查询客户ID为cust1234的完整信息，包括基本信息、历史订单和当前可用优惠。\n\n',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 106,
                        'prompt_tokens': 485,
                        'total_tokens': 591,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 485}
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-plus',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-69911796-eee4-9946-85b2-3eb35f81a508',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--019f6c13-d3c6-7ba0-a2dd-18bc1b988084-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_147a10b37feb4a30a5577463',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_0da75deb7ba744d48be96aff',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_cd6206cb5cfe493d98f9dc5c',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 485,
                    'output_tokens': 106,
                    'total_tokens': 591,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='8f82cc18-abf9-4a3f-8516-febf8db04ce9',
                tool_call_id='call_147a10b37feb4a30a5577463'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='8fe08987-fc00-491f-87c1-2f40cc46070c',
                tool_call_id='call_0da75deb7ba744d48be96aff'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
                name='get_current_promotions',
                id='6f9a4d90-9dd1-4f42-b227-4d991529ab96',
                tool_call_id='call_cd6206cb5cfe493d98f9dc5c'
            )
        ]
    }
}

--------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content='以下是客户ID为cust1234的完整信息：\n\n## 客户基本信息\n- **姓名**：张三\n- 
**等级**：VIP\n- **加入日期**：2023年1月15日\n\n## 订单历史\n- **总订单数**：15单\n- **总花费**：¥25,800.00\n\n## 
当前可用优惠\n- **促销活动**：\n  - 老用户优惠\n  - 会员专属折扣\n- 
**有效期**：至2027年1月31日\n\n该客户是一位VIP级别的优质客户，自2023年加入以来已有15次购买记录，累计消费金额达到25,
800元。目前可以享受老用户优惠和会员专属折扣两项促销活动，有效期至2027年1月31日。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 185,
                        'prompt_tokens': 691,
                        'total_tokens': 876,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 691}
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-plus',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-322bcccb-b2ec-9897-8d68-0db481d7a467',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f6c13-dca0-7d21-8097-a0db879465df-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 691,
                    'output_tokens': 185,
                    'total_tokens': 876,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            )
        ]
    }
}

--------------------------------------------------


## 3、messages输出格式

该模式中会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似 ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。


In [7]:
for chunk in customer_service_agent.stream(
    {
        "messages": [
            {"role":"user", "content":"查询客户id为cust1234的完整信息，历史订单和可用优惠"}
        ]
    },
    stream_mode="messages"
):
    # rprint(chunk)
    # print("-" * 50)
    print(chunk[0].content, end="", flush=True)
    # print("-" * 50)

我将为您查询客户cust1234的完整信息，包括基本信息、历史订单和当前可用优惠。

{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}{"total_orders": 15, "total_spent": 25800.0}{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}以下是客户cust1234的完整信息：

## 客户基本信息
- **姓名**：张三
- **等级**：VIP
- **加入日期**：2023-01-15

## 历史订单信息
- **总订单数**：15单
- **总花费**：¥25,800.00

## 当前可用优惠
- **促销活动**：
  - 老用户优惠
  - 会员专属折扣
- **有效期至**：2027-01-31

该客户是VIP等级的老客户，自2023年1月加入以来已完成了15笔订单，累计消费金额达到25,800元。目前可以享受老用户优惠和会员专属折扣两项促销活动，这些优惠活动将持续到2027年1月31日。

## 4、tasks输出模式

该模式会输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生命周期。


In [ ]:
for chunk in customer_service_agent.stream(
    {
        "messages": [
            {"role":"user", "content":"查询客户id为cust1234的完整信息，历史订单和可用优惠"}
        ]
    },
    stream_mode="tasks"
):
    rprint(chunk)
    print("-" * 50)

5、debug输出模式

该模式与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期。

In [ ]:
for chunk in customer_service_agent.stream(
    {
        "messages": [
            {"role":"user", "content":"查询客户id为cust1234的完整信息，历史订单和可用优惠"}
        ]
    },
    stream_mode="debug"
):
    rprint(chunk)
    print("-" * 50)

## 6、checkpoints输出模式

该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景

In [12]:
from langgraph.checkpoint.memory import InMemorySaver

# 其他工具代码同上，保持不变
# ... ...
# 1. 创建内存检查点存储
checkpointer = InMemorySaver()

# 2. 创建Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions],
    checkpointer=checkpointer  # 启用检查点
)

# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用Agent
checkpoint_count = 0

# 使用checkpoints模式进行流式监控
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        config=config,
        stream_mode="checkpoints"
):
    checkpoint_count += 1
    print(f"检查点 #{checkpoint_count}")
    print(chunk)
    print("-" * 50)

检查点 #1
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f18141f-534f-6e65-bfff-e7ca2a456b17'}}, 'parent_config': None, 'values': {'messages': []}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '8961f9a0-7d2c-d491-5183-863011d0f2a9', 'name': '__start__', 'interrupts': (), 'state': None}]}
--------------------------------------------------
检查点 #2
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f18141f-5355-673f-8000-4b59dfd89af3'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f18141f-534f-6e65-bfff-e7ca2a456b17'}}, 'values': {'messages': [HumanMessage(content='查询客户ID为 CUST123456 的完整信息和可用优惠', additional_kwargs={}, response_metadata={}, id='af1c4ddc-4854-4b38-8829-e7e14d28e062')]}, 'metadata': {'source': 'loop', 'step': 0, 'parents': {}}, 'next': ['model'], 'tasks': [{'id': 'bc1e9958-6b00-

## 7、custom输出模式

开发者通过 get_stream_writer 在工具或节点内部 自定义发送的数据 ，用于 输出 业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [17]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time

@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()
    writer({"type": "生成销售报告", "message": "开始生成销售报告"})
    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告","message": f"生成销售报告进度百分比：{i * 25}%"})
    writer({"type": "生成销售报告", "message": "报告生成完成"})
    return f"销售报告：总收入150万元，同比增长12%"

@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")
    return "当前库存量为10000件，库存充足，无异常"

# 创建报告生成agent
reporting_agent = create_agent(
    model=model,
    tools=[generate_sales_report, generate_inventory_report]
)

for chunk in reporting_agent.stream(
        {"messages": [{"role": "user","content": "帮我生成销售报告和库存报告"}]},
        stream_mode="custom"
):
    print(chunk)
    print("-" * 50)

{'type': '生成销售报告', 'message': '开始生成销售报告'}
--------------------------------------------------
开始库存分析...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：25%'}
--------------------------------------------------
检查当前库存量...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：50%'}
--------------------------------------------------
生成库存报告...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：75%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '报告生成完成'}
--------------------------------------------------


## 8、流式输出模式总结

 LangChain Agent输出模式对比

| 模式 | 输出内容 | 使用场景 |
| ---- | ---- | ---- |
| values | 每个步骤执行后，都会输出完整的状态信息 | 适用于每一步都要获取完整状态、状态持久化场景 |
| updates（默认） | 每个步骤执行后，只增量更新状态中发生变化的内容 | 用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤 |
| messages | 输出流式返回的Token以及相关的元数据（如：来自哪个节点model/tool） | 实现类似ChatGPT 的打字机效果，为聊天机器人等交互式应用提供最佳的实时体验 |
| tasks | 输出当前task任务开始和结束的时间，包含任务的结果和错误信息 | 该模式用于监控任务的生命周期 |
| debug | 与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result） | 该模式用于调试、监控task任务的生命周期 |
| checkpoints | 当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态 | 用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景 |
| custom | 通过get_stream_writer在工具或节点内部自定义发送的数据 | 用于输出业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标 |

我们可以根据不同的目标来选择不同的输出模式。例如：

实现实时对话交互，优先选择**messages**模式；

观察Agent的思考与执行步骤，优先选择**updates**模式；

需要查看每一步状态优先选择**values/tasks/debug**模式；

在工具执行时输出自定义业务日志优先选择**custom**模式。

## 9、组合使用

此外，以上这些模式还可以组合使用，例如，可以同时指定 stream_mode=[“tasks”，“updates”]，这样在同一个循环里既能查看Agent task任务执行内容，又能显示Agent每步的更新。

举例：

In [18]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}

@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }

customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)
for stream_mode, chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        stream_mode=["tasks", "updates"]
):
    print(f"当前流模式: {stream_mode}, 当前数据: {chunk}")
    print("-" * 50)

当前流模式: tasks, 当前数据: {'id': 'e4c959eb-1773-7281-8ac2-27f656673370', 'name': 'model', 'input': {'messages': [HumanMessage(content='查询客户ID为 CUST123456 的完整信息和可用优惠', additional_kwargs={}, response_metadata={}, id='cfd9c9b4-d9ca-44e3-8651-9b84675774fe')]}, 'triggers': ('branch:to:model',)}
--------------------------------------------------
当前流模式: updates, 当前数据: {'model': {'messages': [AIMessage(content='我来为您查询客户ID为 CUST123456 的完整信息和当前可用优惠。\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 485, 'total_tokens': 594, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 485}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-9df73534-31e0-976a-960a-a3e4128c1e88', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f6c2f-b26a-7011-bd59-babe586c52ce-0', tool_calls=[{'name': 'query_customer_data', 'a